# LUMIERE inventory (read-only)

Phase 2, step A1. Lists what is in `MyDrive/LUMIERE` and checks it against LUMIERE's public v202211 rating and completeness tables.

- Reads folder listings and file sizes only. It never opens images and never writes to Drive.
- Prints counts only, without patient or study names. Copy the block after `SUMMARY` back into the chat.
- No GPU needed: a CPU runtime is enough.
- The full listing (with names) stays on this runtime's local disk at `/content/lumiere_inventory.json` and is deleted when the runtime ends.

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-lumiere"
LUMIERE_ROOT = "/content/drive/MyDrive/LUMIERE"
WORKERS = 16  # parallel folder listings on the Drive mount

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
assert Path(LUMIERE_ROOT).is_dir(), f"{LUMIERE_ROOT} not found: check the folder name, or add a shortcut to it in My Drive"
if not Path(WORK).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

In [ ]:
import json

from rhuh_synthesis.lumiere import run_audit

summary = run_audit(LUMIERE_ROOT, workers=WORKERS, inventory_out="/content/lumiere_inventory.json")
print("SUMMARY")
print(json.dumps(summary, indent=1))

## What to send back

Copy everything printed after `SUMMARY`. It holds counts only.

If the last cell stops with an error, send the error message instead. A `FileNotFoundError` means the folder path is wrong. An error from the download step means the two public tables are missing on Drive and could not be fetched from Figshare.